In [1]:


import requests
from bs4 import BeautifulSoup
import pandas as pd
import re
import time
import random
from urllib.parse import urljoin, urlparse, parse_qs, urlencode


from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager


START_URL = "https://www.spinny.com/used-cars-in-hyderabad/s/?utm_medium=gads_c_search&utm_source=SPD-Search-Top8-National-Brand-EM-HYD"
BASE_URL = "https://www.spinny.com"
OUTPUT_CSV = "PROJECT_CARS.csv"
OUTPUT_XLSX = "PROJECT_CARS.xlsx"

MAX_PAGES = 75
DELAY_MIN = 1.0
DELAY_MAX = 2.0
REQUEST_TIMEOUT = 25


EMI_DOWN_PAYMENT_PCT = 0.20
EMI_ANNUAL_RATE = 9.5
EMI_TENURE_YEARS = 5


CARD_SELECTOR = ".CarListingCardV2__carListingCardV2Root"
TITLE_SELECTOR = "a.ListingBrandModelDetail__makeModelLink"
PRICE_SELECTOR = ".ListingBrandModelDetail__priceWithRupeeSymbol"
MORE_DETAILS_SELECTOR = "ul.CarListingCardDetail__more li"
SHORTLIST_SELECTOR = "div.CarListingCardV2__shortlistIcon, div.styles__iconContainer"


def make_session():
    s = requests.Session()
    s.headers.update({
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)",
        "Accept-Language": "en-US,en;q=0.9",
        "Referer": BASE_URL
    })
    return s

def compute_emi(price):
    if not price:
        return None, None, None, None, None

    down = int(price * EMI_DOWN_PAYMENT_PCT)
    loan = price - down
    n = EMI_TENURE_YEARS * 12
    r = EMI_ANNUAL_RATE / 12 / 100

    emi = loan * r * (1+r)**n / ((1+r)**n - 1)
    emi = int(round(emi))

    return down, loan, EMI_ANNUAL_RATE, n, emi

def parse_car_title(title):
    out = {"year": None, "make": "", "model": "", "fuel_type": ""}
    if not title:
        return out
    s = title.strip()

    ym = re.search(r'\b(19|20)\d{2}\b', s)
    if ym:
        out["year"] = int(ym.group())

    s_no_year = re.sub(r'\b(19|20)\d{2}\b', '', s).strip()
    eng = re.search(r'(\d+(\.\d+)?\s*(?:l|L)?(?:\s*[A-Za-z0-9]+)?)$', s_no_year)
    if eng:
        s_no_year = re.sub(re.escape(eng.group(0)) + r'\s*$', '', s_no_year).strip()

    tokens = s_no_year.split()
    if tokens:
        out["make"] = tokens[0]
        if len(tokens) > 1:
            out["model"] = tokens[1]

    lower = title.lower()
    if "diesel" in lower or "crdi" in lower:
        out["fuel_type"] = "Diesel"
    elif "petrol" in lower:
        out["fuel_type"] = "Petrol"
    elif "cng" in lower or "lpg" in lower:
        out["fuel_type"] = "CNG/LPG"
    elif "hybrid" in lower or "electric" in lower or "ev" in lower:
        out["fuel_type"] = "Hybrid/Electric"

    return out

def parse_price_text(text):
    if not text:
        return None
    m_lakh = re.search(r'([\d\.,]+)\s*lakh', text, re.I)
    m_cr = re.search(r'([\d\.,]+)\s*cr', text, re.I)
    if m_lakh:
        return int(float(m_lakh.group(1).replace(',', '')) * 100000)
    if m_cr:
        return int(float(m_cr.group(1).replace(',', '')) * 10000000)
    digits = re.sub(r'[^\d]', '', text)
    return int(digits) if digits else None

def parse_kms_text(text):
    if not text:
        return None
    s = text.lower().replace("km", "").strip()
    m_k = re.search(r'([\d\.,]+)\s*k\b', s)
    if m_k:
        return int(float(m_k.group(1).replace(',', '')) * 1000)
    m_num = re.search(r'([\d,]+)', s)
    if m_num:
        return int(m_num.group(1).replace(',', ''))
    digits = re.sub(r'[^\d]', '', s)
    return int(digits) if digits else None

def parse_card_bs(card):
    d = {
        "year": None, "make": "", "model": "", "fuel_type": "",
        "price_numeric": None, "kms_numeric": None, "transmission": "",
        "down_payment_amount": None, "loan_amount": None,
        "interest_rate_annual": None, "tenure_months": None, "emi_monthly": None
    }

    a = card.select_one(TITLE_SELECTOR)
    title_text = a.get_text(" ", strip=True) if a else ""

    parsed = parse_car_title(title_text)
    d.update(parsed)

    p = card.select_one(PRICE_SELECTOR)
    price_text = p.get_text(" ", strip=True) if p else ""

    sicon = card.select_one(SHORTLIST_SELECTOR)
    data_price_attr = sicon.get("data-price") if sicon and sicon.has_attr("data-price") else None

    d["price_numeric"] = parse_price_text(price_text) or (
        int(data_price_attr) if data_price_attr and str(data_price_attr).isdigit() else None
    )

    details = card.select(MORE_DETAILS_SELECTOR)
    kms_text = fuel_text = trans_text = ""

    if details and len(details) > 0:
        kms_text = details[0].get_text(strip=True)
    if details and len(details) > 1:
        fuel_text = details[1].get_text(strip=True)
    if details and len(details) > 2:
        trans_text = details[2].get_text(strip=True)

    if fuel_text:
        d["fuel_type"] = fuel_text

    d["kms_numeric"] = parse_kms_text(kms_text)
    d["transmission"] = trans_text

    down, loan, rate, n, emi = compute_emi(d["price_numeric"])
    d["down_payment_amount"] = down
    d["loan_amount"] = loan
    d["interest_rate_annual"] = rate
    d["tenure_months"] = n
    d["emi_monthly"] = emi

    return d


def find_next_link(soup, current_url):
    tag = soup.find("link", rel="next")
    if tag and tag.get("href"):
        return urljoin(current_url, tag["href"])

    for a in soup.find_all("a"):
        aria = (a.get("aria-label") or "").lower()
        txt = (a.get_text(" ", strip=True) or "").lower()
        cls = " ".join(a.get("class") or [])

        if "next" in aria or txt in ("next", "›", ">", "→"):
            if a.get("href"):
                return urljoin(current_url, a["href"])

        if "next" in cls and a.get("href"):
            return urljoin(current_url, a["href"])
    return None


def crawl_requests(start_url):
    session = make_session()
    url = start_url
    rows = []
    page = 1

    while url and page <= MAX_PAGES:
        print(f"[requests] Fetching page {page}: {url}")
        try:
            resp = session.get(url, timeout=REQUEST_TIMEOUT)
        except Exception as e:
            print("Requests error:", e)
            break

        if resp.status_code != 200:
            print("Non-200:", resp.status_code)
            break

        soup = BeautifulSoup(resp.text, "html.parser")
        cards = soup.select(CARD_SELECTOR)

        if not cards:
            print("Requests found no cards.")
            break

        for c in cards:
            rows.append(parse_card_bs(c))

        next_link = find_next_link(soup, url)
        if next_link:
            url = next_link
        else:
            parsed = urlparse(url)
            qs = parse_qs(parsed.query)
            cur = int(qs.get("page", ["1"])[0])
            qs["page"] = [str(cur + 1)]
            new_query = urlencode({k: v[0] for k, v in qs.items()})
            url = parsed._replace(query=new_query).geturl()

        page += 1
        time.sleep(random.uniform(DELAY_MIN, DELAY_MAX))

    return rows


def crawl_selenium(start_url):
    print("Starting Selenium fallback...")

    opts = Options()
    opts.add_argument("--headless=new")
    opts.add_argument("--disable-gpu")
    opts.add_argument("--no-sandbox")

    service = Service(ChromeDriverManager().install())
    driver = webdriver.Chrome(service=service, options=opts)

    url = start_url
    rows = []
    page = 1

    try:
        while url and page <= MAX_PAGES:
            print(f"[selenium] Fetching page {page}: {url}")
            driver.get(url)
            time.sleep(4)

            soup = BeautifulSoup(driver.page_source, "html.parser")
            cards = soup.select(CARD_SELECTOR)

            if not cards:
                print("Selenium found no cards.")
                break

            for c in cards:
                rows.append(parse_card_bs(c))

            next_link = find_next_link(soup, url)
            if next_link:
                url = next_link
            else:
                parsed = urlparse(url)
                qs = parse_qs(parsed.query)
                cur = int(qs.get("page", ["1"])[0])
                qs["page"] = [str(cur + 1)]
                new_query = urlencode({k: v[0] for k, v in qs.items()})
                url = parsed._replace(query=new_query).geturl()

            page += 1
    finally:
        driver.quit()

    return rows


def run_all():
    print("Trying requests...")
    rows = crawl_requests(START_URL)

    if not rows:
        print("Falling back to Selenium...")
        rows = crawl_selenium(START_URL)

    if not rows:
        print("No data collected.")
        return

    df = pd.DataFrame(rows)

    cols = [
        "year", "make", "model", "fuel_type",
        "price_numeric", "kms_numeric", "transmission",
        "down_payment_amount", "loan_amount",
        "interest_rate_annual", "tenure_months", "emi_monthly"
    ]

    for c in cols:
        if c not in df.columns:
            df[c] = None

    df = df[cols]

  
    df = df.rename(columns={
        "year": "Year",
        "make": "Brand",
        "model": "Model",
        "fuel_type": "Fuel Type",
        "price_numeric": "Price (₹)",
        "kms_numeric": "Kilometers Driven",
        "transmission": "Transmission",
        "down_payment_amount": "Down Payment (₹)",
        "loan_amount": "Loan Amount (₹)",
        "interest_rate_annual": "Interest Rate (%)",
        "tenure_months": "Tenure (Months)",
        "emi_monthly": "EMI (₹ / Month)"
    })

    df.to_csv(OUTPUT_CSV, index=False)
    df.to_excel(OUTPUT_XLSX, index=False)

    print(f"Saved {len(df)} rows → {OUTPUT_CSV}, {OUTPUT_XLSX}")

if __name__ == "__main__":
    run_all()


Trying requests...
[requests] Fetching page 1: https://www.spinny.com/used-cars-in-hyderabad/s/?utm_medium=gads_c_search&utm_source=SPD-Search-Top8-National-Brand-EM-HYD
Requests found no cards.
Falling back to Selenium...
Starting Selenium fallback...
[selenium] Fetching page 1: https://www.spinny.com/used-cars-in-hyderabad/s/?utm_medium=gads_c_search&utm_source=SPD-Search-Top8-National-Brand-EM-HYD
[selenium] Fetching page 2: https://www.spinny.com/used-cars-in-hyderabad/s/?utm_medium=gads_c_search&utm_source=SPD-Search-Top8-National-Brand-EM-HYD&page=2
[selenium] Fetching page 3: https://www.spinny.com/used-cars-in-hyderabad/s/?utm_medium=gads_c_search&utm_source=SPD-Search-Top8-National-Brand-EM-HYD&page=3
[selenium] Fetching page 4: https://www.spinny.com/used-cars-in-hyderabad/s/?utm_medium=gads_c_search&utm_source=SPD-Search-Top8-National-Brand-EM-HYD&page=4
[selenium] Fetching page 5: https://www.spinny.com/used-cars-in-hyderabad/s/?utm_medium=gads_c_search&utm_source=SPD-Searc